# Stage 12 — Paper trading journal (debit spread + option buying, forward-only)

Neither strategy has been traded with real money yet. Both already have a fully
backtested trade lifecycle (Stages 9/11) on real historical data since 2023 —
what hasn't been checked is whether that same signal keeps producing a similar
outcome distribution **going forward**, on trades nobody hand-picked with
hindsight.

**No new entry/exit rule is added here.** This refreshes the exact same
shortlist notebooks 16/19 already refresh, using the same
`process_instrument_incremental()` + trade-lifecycle recompute those notebooks
use — then keeps every trade whose `entry_date` falls on or after this
journal's start date (fixed the first time this ran, saved to
`config/paper_trading_start.json`, never moved afterward).

**Still entirely paper.** Nothing here places, closes, or modifies any real
order — this only journals what the signal would have done, so the results can
be reviewed after a month or two (win rate, avg R, total R — same metrics the
historical backtests already use) before risking real capital.

Run this once a day — cheap, incremental fetch only. Re-running more often just
refreshes OPEN trades' current unrealized R, no harm.

In [1]:
import sys
sys.path.append("..")

import pandas as pd

ranking_debit = pd.read_excel("../exports/RANKING_debit_spread.xlsx")
ranking_option = pd.read_excel("../exports/RANKING_option_buying.xlsx")

# Same shortlist rule notebooks 16/19 already use — outlier-reliance-clean,
# top 20 by score, NIFTY/BANKNIFTY always included if clean. Not a new
# selection rule, just reused so the paper journal watches exactly what the
# live screeners already consider tradeable.
def shortlist(ranking):
    clean = ranking[ranking["outlier_reliance_flag"].isna() | (ranking["outlier_reliance_flag"] == "")]
    top_20 = clean.head(20)
    must_include = ranking[ranking["symbol"].isin(["NIFTY", "BANKNIFTY"])]
    must_include_clean = must_include[must_include["outlier_reliance_flag"].isna() | (must_include["outlier_reliance_flag"] == "")]
    return pd.concat([top_20, must_include_clean]).drop_duplicates(subset="symbol").reset_index(drop=True)

debit_candidates = shortlist(ranking_debit)
option_candidates = shortlist(ranking_option)

print(f"Debit spread shortlist: {len(debit_candidates)} symbols")
print(f"Option buying shortlist: {len(option_candidates)} symbols")

Debit spread shortlist: 20 symbols
Option buying shortlist: 21 symbols


In [2]:
from datetime import datetime
from src.auth import get_kite
from src.instruments import build_nse_token_lookup, get_index_token
from src.debit_spread_scale_pipeline import process_instrument_incremental as refresh_debit_spread
from src.option_buying_scale_pipeline import process_instrument_incremental as refresh_option_buying

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)
index_symbols = {"NIFTY": ("NIFTY 50", "NSE"), "BANKNIFTY": ("NIFTY BANK", "NSE")}

def token_for(symbol):
    if symbol in index_symbols:
        tradingsymbol, exchange = index_symbols[symbol]
        return get_index_token(kite, tradingsymbol, exchange)
    return token_lookup.get(symbol)

def refresh(symbols, refresh_fn, label):
    for i, symbol in enumerate(symbols, start=1):
        try:
            token = token_for(symbol)
            if token is None:
                print(f"[{label} {i}/{len(symbols)}] {symbol} — no token found, skipping")
                continue
            refresh_fn(symbol, token, kite, to_date=datetime.now())
            print(f"[{label} {i}/{len(symbols)}] {symbol} — refreshed")
        except Exception as e:
            print(f"[{label} {i}/{len(symbols)}] {symbol} — FAILED: {e}")

refresh(debit_candidates["symbol"].tolist(), refresh_debit_spread, "debit spread")
refresh(option_candidates["symbol"].tolist(), refresh_option_buying, "option buying")

Fetching 2026-05-10 to 2026-09-08...
Saved 6327 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/POWERINDIA_60minute.csv
Saved 68 trades to /Users/hgmac/Documents/3_trading/Quant/exports/debit_spread/POWERINDIA_trades.csv
[debit spread 1/20] POWERINDIA — refreshed
Fetching 2026-05-10 to 2026-09-08...
Saved 6327 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/TIINDIA_60minute.csv
Saved 67 trades to /Users/hgmac/Documents/3_trading/Quant/exports/debit_spread/TIINDIA_trades.csv
[debit spread 2/20] TIINDIA — refreshed
Fetching 2026-05-10 to 2026-09-08...
Saved 6327 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/CDSL_60minute.csv
Saved 69 trades to /Users/hgmac/Documents/3_trading/Quant/exports/debit_spread/CDSL_trades.csv
[debit spread 3/20] CDSL — refreshed
Fetching 2026-05-10 to 2026-09-08...
Saved 6327 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/PFC_60minute.csv
Saved 63 trades to /Users/hgmac/Documents/3_trading/Quant/exports/debit_spread

## The journal — forward trades only, since tracking started

In [3]:
from src.paper_trading import build_journal, summarize, save_journal, get_start_date

debit_journal = build_journal(debit_candidates["symbol"].tolist(), "debit_spread")
option_journal = build_journal(option_candidates["symbol"].tolist(), "option_buying")

journal = pd.concat([debit_journal, option_journal], ignore_index=True)

print(f"Debit spread paper-tracking since {get_start_date('debit_spread').date()}")
print(f"Option buying paper-tracking since {get_start_date('option_buying').date()}")

if journal.empty:
    print("\nNo forward trades yet — nothing has fired since tracking started. Re-run daily.")
else:
    path = save_journal(journal)
    print(f"\n{len(journal)} forward trade(s) so far, saved to {path}")

Debit spread paper-tracking since 2026-09-07
Option buying paper-tracking since 2026-09-07

2 forward trade(s) so far, saved to /Users/hgmac/Documents/3_trading/Quant/logs/paper_trading_journal.csv


In [4]:
if not journal.empty:
    summary = summarize(journal)
    print("SUMMARY — by strategy and status (OPEN = live unrealized R, CLOSED = resolved):")
    display(summary)

    print("\nFull journal, most recent entry first:")
    display(journal.sort_values("entry_date", ascending=False))

SUMMARY — by strategy and status (OPEN = live unrealized R, CLOSED = resolved):


,strategy,status,n,win_rate_pct,avg_r,median_r,total_r
0,debit_spread,OPEN,1,0.0,-0.44,-0.44,-0.44
1,option_buying,OPEN,1,100.0,0.49,0.49,0.49



Full journal, most recent entry first:


,strategy,symbol,direction,status,entry_date,entry_price,stop_price,target_price,exit_date,exit_price,exit_reason,candles_held,pct_move,r_achieved,max_adverse_pct,initial_stop_price,final_stop_price,trailing_activated,four_h_direction
1,option_buying,GODREJPROP,DOWN,OPEN,2026-09-07 11:15:00,1920.4,NaN,NaN,2026-09-08 10:15:00,1901.8,OPEN,5,0.97,0.49,0.5,1958.15,1958.15,False,DOWN
0,debit_spread,MCX,UP,OPEN,2026-09-07 09:15:00,3342.1,3278.07,3438.14,2026-09-08 10:15:00,3313.8,OPEN,7,-0.85,-0.44,1.2,NaN,NaN,NaN,NaN
